# [*Problema do caixeiro viajante*](http://en.wikipedia.org/wiki/Traveling_salesman_problem) (TSP):

> *Dado um conjunto de cidades e as distâncias entre cada par de cidades, encontre um **tour** das cidades com a distância total mínima. Um **tour** significa que você começa em uma cidade, visita todas as outras cidades exatamente uma vez e depois retorna à cidade inicial.*

- Este notebook se baseia no [Notebook IPython sobre o problema do caixeiro viajante] de [Peter Norvig](http://norvig.com/)(http://nbviewer.ipython.org/url/norvig.com/ipython/TSPv3 .ipynb).
- Mostrarei como aplicar algoritmos evolutivos para resolver o TSP.

- Este é um problema [*intratável*](http://en.wikipedia.org/wiki/Intractability_(complexity) bem conhecido, o que significa que não existem soluções eficientes que funcionem para um grande número de cidades.
- Podemos criar um algoritmo ineficiente que funcione bem para um pequeno número de citações (cerca de uma dúzia).
- Também podemos encontrar um passeio *quase* mais curto por milhares de cidades.
- Na verdade, o fato de não existir um algoritmo eficiente é libertador:
> **Isso significa que podemos usar um algoritmo muito simples e ineficiente e não nos sentirmos mal por isso.**

### O *vocabulário* do problema:

- **Cidade**: Para efeitos deste exercício, uma cidade é "atômica" no sentido de que não precisamos saber nada sobre os componentes ou atributos de uma cidade, apenas a que distância ela está de outras cidades.
- **Cidades**: Precisaremos representar um conjunto de cidades; O tipo de dados `set` do Python pode ser apropriado para isso.
- **Distância**: Precisaremos da distância entre duas cidades. Se `A` e `B` são cidades. Isso pode ser feito com uma função, `distance(A, B)`, ou com um dict, `distance[A][B]` ou `distance[A, B]`, ou com um array se `A` e `B` são índices inteiros. A distância resultante será um número real (que Python chama de `float`).
- **Tour**: Um tour é uma lista ordenada de cidades; Os tipos de dados `list` ou `tuple` do Python funcionariam.
- **Distância total**: soma das distâncias das cidades adjacentes no passeio. Provavelmente teremos uma função, `total_distance(tour)`.

In [33]:
import numpy as np
import random
from pulp import *
import networkx as nx

class TSPProblem:
    def __init__(self, custo):
        self.custo = custo
        self.n = len(custo)
        self.arcos = [(i, j) for i in range(self.n) for j in range(self.n) if i != j]
        self.tsp = LpProblem("CaixeiroViajante", LpMinimize)

    def build_problem(self):
        # Variáveis de decisão
        x = LpVariable.dicts("x", self.arcos, cat="Binary")
        
        # Função objetivo
        self.tsp += lpSum([self.custo[i][j] * x[(i, j)] for (i, j) in self.arcos])

        # Restrições
        for i in range(self.n):
            self.tsp += lpSum([x[(i, j)] for j in range(self.n) if i != j]) == 1
            self.tsp += lpSum([x[(j, i)] for j in range(self.n) if i != j]) == 1

        # Resolver o problema
        self.tsp.solve()

    def get_solution(self):
        solucao = []
        for v in self.tsp.variables():
            if v.varValue > 0 and v.name.startswith("x"):
                solucao.append(v.name)
        return solucao

# Número de cidades
n_cidades = 4

# Gerar matriz de custos

#Lista com os arcos atravessados pelo caixeiro
arcos_caminho = []


custo = np.random.randint(100, 1000, size=(n_cidades, n_cidades))
print(custo)

#Parametros
n = len(custo)
arcos = [(i,j) for i in range(n) for j in range(n) if custo[i,j] != 999]

np.fill_diagonal(custo, 999)  # Substituir diagonal por um valor alto (sem conexão)

# Resolver o problema TSP
tsp_problem = TSPProblem(custo)
tsp_problem.build_problem()

# Obter a solução
solucao = tsp_problem.get_solution()
print("Solução do TSP:")
print(solucao)

#Criar o grafo
G = nx.Graph()

#Adicionando todos os arcos
G.add_edges_from(arcos)

#Posicao dos nos
coordenadas = [[2,3], [3, 2], [3,1], [1,1], [1,2]]
#Lista com os arcos atravessados pelo caixeiro
arcos_caminho = []

for var in tsp_problem.tsp.variables():
    if var.name[0] == "x":
        if var.varValue > 0:
            #Encontrando os indices para dividir a string
            open_parenthesis = var.name.find("(")
            close_parenthesis = var.name.find(")")
            comma = var.name.find(",")
            second_underline = var.name.find("_", comma)
            
            #Determinar (i,j)
            i = int(var.name[open_parenthesis+1: comma])
            j = int(var.name[second_underline+1: close_parenthesis])
            
            arcos_caminho.append((i,j))
            arcos_caminho.append((j,i)) #Simetrico
            
print(arcos_caminho)

#Identificando a cor dos arcos
cores = []
for arco in arcos:
    if arco in arcos_caminho:
        cores.append("red")
    else:
        cores.append("black")

#Plotar o grafo
nx.draw(G, pos = coordenadas, with_labels = True, node_color = "white", 
        edgecolors = "black", edgelist = arcos, edge_color = cores)


[[341 700 228 191]
 [820 885 118 988]
 [829 690 199 804]
 [581 120 667 310]]
Solução do TSP:
['x_(0,_3)', 'x_(1,_2)', 'x_(2,_0)', 'x_(3,_1)']
[(0, 3), (3, 0), (1, 2), (2, 1), (2, 0), (0, 2), (3, 1), (1, 3)]
Error in callback <function _draw_all_if_interactive at 0x000001ECD844C360> (for post_execute):


RuntimeError: Failed to process string with tex because latex could not be found

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 640x480 with 1 Axes>

[[647 410 165 708]
 [522 466 858 698]
 [744 452 885 955]
 [875 406 108 461]]
Solução do TSP:
['x_(0,_3)', 'x_(1,_0)', 'x_(2,_1)', 'x_(3,_2)']
[(0, 3), (3, 0), (1, 0), (0, 1), (2, 1), (1, 2), (3, 2), (2, 3)]
Error in callback <function _draw_all_if_interactive at 0x000001ECD844C360> (for post_execute):


RuntimeError: Failed to process string with tex because latex could not be found

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 640x480 with 1 Axes>

In [ ]:
# Função para gerar cidades
def generate_cities(n):
    custo = np.random.randint(100, 1000, size=(n, n))
    np.fill_diagonal(custo, 999)
    return custo

# Função para plotar o gráfico
def plot_graph(arcos, arcos_caminho, coordenadas, title):
    G = nx.Graph()
    G.add_edges_from(arcos)
    cores = ["red" if arco in arcos_caminho or (arco[1], arco[0]) in arcos_caminho else "black" for arco in arcos]
    nx.draw(G, pos=coordenadas, with_labels=True, node_color="white", edgecolors="black", edgelist=arcos, edge_color=cores)
    plt.title(title)
    plt.show()

# Gerar e resolver o TSP para três conjuntos de dados
for i in range(1, 4):
    n_cidades = 5  # Número de cidades
    custo = generate_cities(n_cidades)
    tsp_problem = TSPProblem(custo)
    tsp_problem.build_problem()
    solucao = tsp_problem.get_solution()

    # Preparar dados para plotagem
    arcos = [(i, j) for i in range(n_cidades) for j in range(n_cidades) if i != j]
    arcos_caminho = [(int(var.name.split("_")[1].split(",")[0]), int(var.name.split(",")[1].split(")")[0])) for var in tsp_problem.tsp.variables() if var.varValue > 0 and var.name.startswith("x")]
    coordenadas = {i: (random.randint(0, 10), random.randint(0, 10)) for i in range(n_cidades)}

    # Plotar o gráfico
    plot_graph(arcos, arcos_caminho, coordenadas, f"TSP Solution {i}")

In [32]:
# Exemplo para três instâncias do TSP
for i in range(1, 4):
    custo = generate_cities(5)  # Gerar 5 cidades para cada instância
    np.fill_diagonal(custo, 999)
    tsp_problem = TSPProblem(custo)
    tsp_problem.build_problem()
    solucao = tsp_problem.get_solution()
    print(f"Solução do TSP {i}:")
    print(solucao)
    # Aqui você adicionaria o código para extrair os arcos da solução e plotar o grafo


AttributeError: 'set' object has no attribute 'ndim'

## Criando as cidades

In [18]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cmx

import random, operator, time, itertools, math
import numpy

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import seaborn
seaborn.set(style='whitegrid')
seaborn.set_context('notebook')

alltours = itertools.permutations # The permutation function is already defined in the itertools module
cities = {1, 2, 3}
list(alltours(cities))

City = complex # Constructor for new cities, e.g. City(300, 400)


### functions

In [24]:

def distance(A, B): 
    "The Euclidean distance between two cities."
    return abs(A - B)

def generate_cities(n):
    "Make a set of n cities, each with random coordinates."
    return {City(random.randrange(10, 890), random.randrange(10, 590)) for c in range(n)}

def total_distance(tour):
    "The total distance between each pair of consecutive cities in the tour."
    return sum(distance(tour[i], tour[i-1]) for i in range(len(tour)))

def exact_TSP(cities):
    "Generate all possible tours of the cities and choose the shortest one."
    return shortest(itertools.permutations(cities))

def shortest(tours): 
    "Return the tour with the minimum total distance."
    return min(tours, key=total_distance)

def plot_tour(tour): 
    "Plot the tour on a 2-D plane."
    plotline(list(tour) + [tour[0]])
    plotline([tour[0]], style='gD', size=10)
    plt.show()

def plotline(points, style='bo-', size=7):
    "Plot a list of points (complex numbers) in the 2-D plane."
    X, Y = XY(points)
    plt.plot(X, Y, style, markersize=size)
    
def XY(points):
    "Given a list of points, return two lists: X coordinates, and Y coordinates."
    return [p.real for p in points], [p.imag for p in points]



### Exemplo de uso

In [20]:

A = City(300, 0)
B = City(0, 400)
distance(A, B)

cities8, cities10, cities100, cities1000 = generate_cities(8), generate_cities(10), generate_cities(100), generate_cities(1000)
print("Cidade = ",cities8)

Cidade =  {(532+26j), (775+554j), (220+582j), (758+436j), (869+390j), (782+153j), (411+565j), (831+32j)}


In [21]:
# Gerar cidades e resolver o TSP
cities8 = generate_cities(8)
tour = exact_TSP(cities8)
plot_tour(tour)

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 640x480 with 1 Axes>

# 3

In [25]:
class TSPSolver:
    def __init__(self, cities):
        self.cities = cities

    def exactTSP(self):
        return self.shortest(itertools.permutations(self.cities))

    def totalDistance(self, tour):
        return sum(distance(tour[i], tour[i-1]) for i in range(len(tour)))

    def shortest(self, tours):
        return min(tours, key=self.totalDistance)

class CityGenerator:
    def __init__(self, numberOfCities):
        self.numberOfCities = numberOfCities

    def generateCities(self):
        return {City(random.randrange(10, 890), random.randrange(10, 590)) for _ in range(self.numberOfCities)}

def plotTour(tour): 
    plotLine(list(tour) + [tour[0]])
    plotLine([tour[0]], style='gD', size=10)
    plt.show()

def plotLine(points, style='bo-', size=7):
    X, Y = xy(points)
    plt.plot(X, Y, style, markersize=size)

def xy(points):
    return [p.real for p in points], [p.imag for p in points]

def distance(A, B): 
    return abs(A - B)


In [28]:

A = City(300, 0)
B = City(0, 400)
distance(A, B)

cities8, cities10, cities100, cities1000 = generate_cities(8), generate_cities(10), generate_cities(100), generate_cities(1000)
print("Cidade = ",cities8)

# Gerar cidades
cityGenerator = CityGenerator(8)
#cities8 = cityGenerator.generateCities()

# Resolver o TSP
tspSolver = TSPSolver(cities8)
tour = tspSolver.exactTSP()

# Plotar o tour
plotTour(tour)


Cidade =  {(594+59j), (137+283j), (428+53j), (106+54j), (123+115j), (331+378j), (116+418j), (414+106j)}


RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 640x480 with 1 Axes>